# Creating summary table for all variables
- table - app_train.csv, app_valid.csv
---

## 0. Libraries and data

In [1]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path
import openpyxl

PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / 'src'

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from eda_module import (
    plot_quantitative_distribution, plot_binary_distribution, 
    plot_binary_vs_binary, plot_quantitative_vs_binary, plot_categorical_distribution,
    plot_categorical_vs_binary
)

from preprocess_module import (
    bin_quantitative_var, fit_optbin_var, fit_quantitative_binner, transform_quantitative_binner,
    fit_capper, transform_capper, create_imputed_quantitative_features
)

In [2]:
df_train = pd.read_csv(r"..\..\data\interim\app_train.csv")
print(f"Shape of df_train: {df_train.shape}")
df_train.head(10)

Shape of df_train: (215257, 122)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,TARGET
0,285133,Cash loans,F,Y,Y,2,405000.0,1971072.0,68643.0,1800000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,191894,Cash loans,M,N,Y,0,337500.0,508495.5,38146.5,454500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0,0
2,369428,Cash loans,M,N,Y,1,112500.0,110146.5,13068.0,90000.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,138717,Cash loans,F,N,Y,2,40500.0,66384.0,3519.0,45000.0,...,0,0,0,0.0,0.0,0.0,1.0,0.0,2.0,0
4,202381,Cash loans,M,Y,N,0,225000.0,298512.0,31801.5,270000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
5,185763,Cash loans,F,N,Y,0,315000.0,417024.0,21964.5,360000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0,0
6,148765,Cash loans,M,Y,Y,0,162000.0,168102.0,11362.5,148500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0
7,448028,Cash loans,F,N,Y,0,135000.0,1129500.0,33025.5,1129500.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
8,428649,Cash loans,F,N,Y,0,157500.0,1260702.0,39712.5,1129500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0
9,345767,Cash loans,F,Y,N,0,180000.0,879480.0,25843.5,630000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0,0


In [3]:
df_valid = pd.read_csv(r"..\..\data\interim\app_valid.csv")
print(f"Shape of df_valid: {df_valid.shape}")
df_valid.head(10)

Shape of df_valid: (92254, 122)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,TARGET
0,331034,Cash loans,F,N,Y,0,90000.0,803259.0,23616.0,670500.0,...,0,0,0,0.0,0.0,0.0,1.0,0.0,1.0,0
1,242581,Cash loans,F,N,Y,0,270000.0,1288350.0,41692.5,1125000.0,...,0,0,0,0.0,0.0,0.0,1.0,0.0,4.0,0
2,281583,Cash loans,F,N,Y,0,270000.0,253737.0,25227.0,229500.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,255865,Cash loans,F,N,Y,0,144000.0,436032.0,16564.5,360000.0,...,0,0,0,0.0,0.0,1.0,0.0,0.0,2.0,0
4,389379,Revolving loans,M,N,N,2,72000.0,225000.0,11250.0,225000.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
5,455506,Cash loans,M,Y,Y,0,315000.0,521280.0,27423.0,450000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0,0
6,234885,Cash loans,F,Y,Y,1,54000.0,102384.0,6673.5,81000.0,...,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0,0
7,425217,Cash loans,M,Y,N,0,315000.0,545040.0,31288.5,450000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0,0
8,209875,Cash loans,F,N,Y,1,315000.0,254700.0,13567.5,225000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
9,106521,Cash loans,F,Y,N,0,67500.0,292500.0,13014.0,292500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0


In [4]:
numeric_cols = list(df_train.select_dtypes(include=['Int64', 'float64']).columns)
cat_cols = list(df_train.select_dtypes(include=['O']).columns)

print((len(numeric_cols) + len(cat_cols)) == df_train.shape[1])

True


## 1. Optimal binning

In [ ]:
vars_to_bin = [col for col in df_train.columns if col != "TARGET" and col != 'SK_ID_CURR']

summary_list = []

for col in vars_to_bin:
    df_train, df_valid, optb_n, summary_n = fit_optbin_var(
        df_train=df_train,
        df_valid=df_valid,
        var=col,
        dtype="numerical" if col in numeric_cols else "categorical",
        target="TARGET",
        metric="bins",
        monotonic_trend="auto" if col in numeric_cols else None,
        min_bin_size=0.05,
        special_codes=[365243] if col == "DAYS_EMPLOYED" else None,
        missing_as_category = True if col in ['AMT_REQ_CREDIT_BUREAU_YEAR', 'OCCUPATION_TYPE']
    )
    summary_list.append(summary_n)

summary = pd.concat(summary_list, axis=0, ignore_index=True)

In [6]:
summary.shape

(811, 18)

In [7]:
summary

,Variable,Type,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS,Count_valid,Count (%)_valid,Non-event_valid,Event_valid,Event rate_valid,PSI,PSI_total
0,NAME_CONTRACT_TYPE_optbin,categorical,[Revolving loans],20582.0,0.095616,19440.0,1142.0,0.055485,0.402038,0.013075,1.623483e-03,8697.0,0.094272,8235.0,462.0,0.053122,0.000019,0.000021
1,NAME_CONTRACT_TYPE_optbin,categorical,[Cash loans],194675.0,0.904384,178440.0,16235.0,0.083395,-0.03543,0.001152,1.440268e-04,83557.0,0.905728,76571.0,6986.0,0.083608,0.000002,0.000021
2,NAME_CONTRACT_TYPE_optbin,categorical,Special,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,NaN,0.000000,0.000021
3,NAME_CONTRACT_TYPE_optbin,categorical,Missing,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,NaN,0.000000,0.000021
4,NAME_CONTRACT_TYPE,categorical,Totals,215257.0,1.000000,197880.0,17377.0,0.080727,NaN,0.014227,1.767510e-03,92254.0,1.000000,84806.0,7448.0,0.080734,NaN,0.000021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
806,AMT_REQ_CREDIT_BUREAU_YEAR_optbin,numerical,"[3.50, 4.50)",14555.0,0.067617,13375.0,1180.0,0.081072,-0.00464,0.000001,1.823592e-07,6159.0,0.066761,5629.0,530.0,0.086053,0.000011,0.000025
807,AMT_REQ_CREDIT_BUREAU_YEAR_optbin,numerical,"[4.50, inf)",18320.0,0.085108,16724.0,1596.0,0.087118,-0.083169,0.000610,7.617824e-05,7932.0,0.085980,7222.0,710.0,0.089511,0.000009,0.000025
808,AMT_REQ_CREDIT_BUREAU_YEAR_optbin,numerical,Special,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,NaN,0.000000,0.000025
809,AMT_REQ_CREDIT_BUREAU_YEAR_optbin,numerical,Missing,NaN,NaN,NaN,NaN,NaN,-0.287634,0.012611,1.570993e-03,NaN,NaN,NaN,NaN,NaN,0.000000,0.000025


In [8]:
summary.sort_values(by=['Variable', 'WoE'], ascending=[True, False]).head(20)

,Variable,Type,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS,Count_valid,Count (%)_valid,Non-event_valid,Event_valid,Event rate_valid,PSI,PSI_total
62,AMT_ANNUITY,numerical,Totals,215257.0,1.000000,197880.0,17377.0,0.080727,NaN,0.031558,0.003929,92254.0,1.000000,84806.0,7448.0,0.080734,NaN,0.000223
59,AMT_ANNUITY_optbin,numerical,"[46671.75, inf)",20247.0,0.094060,19106.0,1141.0,0.056354,0.385584,0.011911,0.001480,8606.0,0.093286,8110.0,496.0,0.057634,6.391002e-06,0.000223
51,AMT_ANNUITY_optbin,numerical,"(-inf, 12728.25)",29794.0,0.138411,27755.0,2039.0,0.068437,0.178155,0.004077,0.000509,12734.0,0.138032,11868.0,866.0,0.068007,1.041069e-06,0.000223
58,AMT_ANNUITY_optbin,numerical,"[41627.25, 46671.75)",10882.0,0.050554,10131.0,751.0,0.069013,0.169436,0.001352,0.000169,4632.0,0.050209,4292.0,340.0,0.073402,2.353177e-06,0.000223
52,AMT_ANNUITY_optbin,numerical,"[12728.25, 16404.75)",22969.0,0.106705,21284.0,1685.0,0.073360,0.103677,0.001098,0.000137,9903.0,0.107345,9185.0,718.0,0.072503,3.826325e-06,0.000223
60,AMT_ANNUITY_optbin,numerical,Special,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000223
61,AMT_ANNUITY_optbin,numerical,Missing,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000223
57,AMT_ANNUITY_optbin,numerical,"[37199.25, 41627.25)",12607.0,0.058567,11554.0,1053.0,0.083525,-0.037125,0.000082,0.000010,5678.0,0.061547,5262.0,416.0,0.073265,1.479221e-04,0.000223
56,AMT_ANNUITY_optbin,numerical,"[31637.25, 37199.25)",22854.0,0.106171,20929.0,1925.0,0.084230,-0.046304,0.000232,0.000029,9607.0,0.104136,8722.0,885.0,0.092120,3.935899e-05,0.000223
53,AMT_ANNUITY_optbin,numerical,"[16404.75, 26507.25)",66931.0,0.310935,61082.0,5849.0,0.087389,-0.086567,0.002416,0.000302,28753.0,0.311672,26275.0,2478.0,0.086182,1.743978e-06,0.000223


## 2. Saving report (optimal bins)

In [11]:
from pathlib import Path
import sys
import pandas as pd
from openpyxl.utils import get_column_letter

PROJECT_PATH = Path.cwd().parent.parent
SUMMARY_SAVE_PATH = PROJECT_PATH / "reports"

if str(SUMMARY_SAVE_PATH) not in sys.path:
    sys.path.insert(0, str(SUMMARY_SAVE_PATH))

SUMMARY_SAVE_PATH.mkdir(parents=True, exist_ok=True)

file_path = SUMMARY_SAVE_PATH / "optbin_summary.xlsx"

summary_export = summary.copy()
summary_export["Bin"] = summary_export["Bin"].astype(str)

with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
    summary_export.to_excel(writer, sheet_name="summary", index=False)

    ws = writer.book["summary"]

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for col_idx, col_name in enumerate(summary_export.columns, start=1):
        max_len = max(
            len(str(col_name)),
            summary_export.iloc[:, col_idx - 1].astype(str).map(len).max()
        )
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 2, 40)

print(f"Saved to: SUMMARY_SAVE_PATH/optbin_summary.xlsx")

Saved to: SUMMARY_SAVE_PATH/optbin_summary.xlsx
